# Verify attention and transformer shape contracts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thehalleyyoung/tensorguard/blob/main/examples/tutorials/06_attention.ipynb)

Scaled-dot-product attention is a real PyTorch kernel with multi-axis contracts: query/key embedding dims must agree and key/value sequence lengths must agree, while value width controls the output width. TensorGuard checks those facts statically.

In [ ]:
%pip install -q tensorguard  # on Colab; locally: pip install -e .

In [ ]:
import torch, torch.nn as nn
import torch.nn.functional as F
from src.fx_extractor import verify_module

class AttentionBlock(nn.Module):
    def forward(self, q, k, v):
        return F.scaled_dot_product_attention(q, k, v)

good = verify_module(AttentionBlock(),
                     input_shapes={'q': (2, 4, 5, 8),
                                   'k': (2, 4, 7, 8),
                                   'v': (2, 4, 7, 9)})
print('good attention safe:', good.safe)
assert good.safe

A key/value sequence mismatch is caught before the kernel runs:

In [ ]:
bad = verify_module(AttentionBlock(),
                    input_shapes={'q': (2, 4, 5, 8),
                                  'k': (2, 4, 7, 8),
                                  'v': (2, 4, 6, 9)})
print('bad attention safe:', bad.safe)
assert not bad.safe
assert any(v.kind == 'shape_incompatible'
           for v in bad.counterexample.violations)